# Mutation Walkthrough

This notebook walks through the current mutation pipeline in `tree_diffusion_integration`:

`prefix expression -> AST -> canonical form -> position index -> local replacement / sampled subtree replacement -> mutate_once(...)`

The most important functions in that pipeline are `parse_prefix_string(...)`, `serialize_prefix_string(...)`, `canonicalize(...)`, `index_tree_positions(...)`, `local_replacement_candidates(...)`, `can_locally_replace(...)`, `local_replace_once(...)`, `sample_valid_subtree(...)`, `can_sampled_subtree_replace(...)`, `replace_subtree_by_node_id(...)`, `collect_candidate_nodes(...)`, and `mutate_once(...)`.

The goal is to make each stage concrete: what each function takes as input, what it returns, what invariants it enforces, and why the later mutation stages depend on the earlier normalization steps.

The notebook is written as a runnable tutorial and intentionally stores no executed outputs.


In [1]:
from pathlib import Path
import sys
import random
from fractions import Fraction
from pprint import pprint
from dataclasses import asdict


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").is_dir() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not find the tree_diffusion_integration repo root from the current working directory."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Using repo root: {REPO_ROOT}")

from src.mathlang.ast import BinaryOp, Const, NaryOp, UnaryOp, Var
from src.mathlang.parser import parse_prefix_string
from src.mathlang.serializer import serialize_prefix_string
from src.mathlang.canonicalize import canonicalize
from src.tree_diffusion.positions import index_tree_positions
from src.tree_diffusion.mutation_grammar import (
    can_locally_replace,
    can_sampled_subtree_replace,
    local_replacement_candidates,
)
from src.tree_diffusion.mutation import (
    collect_candidate_nodes,
    local_replace_once,
    mutate_once,
    replace_subtree_by_node_id,
    sample_valid_subtree,
)


Using repo root: /workspace/rbarket/tree_diffusion_integration


## Example Expressions And ASTs

We start with a few concrete prefix expressions, parse them into the math AST classes, and print both the raw dataclass representation and a small recursive tree view.

`parse_prefix_string(...)` is the entry point from notebook-friendly text into the typed AST used everywhere else in the mutation code. It tokenizes on whitespace, recognizes `INT+` / `INT-` followed by digit tokens as numeric constants, parses unary operators into `UnaryOp`, parses `pow` / `div` into `BinaryOp`, and parses `add` / `mul` into `NaryOp` by recursively collecting same-operator operands.

`serialize_prefix_string(...)` is the inverse view we use throughout the walkthrough. It turns the AST back into the prefix token stream that the notebook prints in before / after comparisons, so it is the easiest way to see how a mutation changed structure. Numeric constants round-trip through the serializer's token format, including fractions, and a parsed constant `div` can already collapse to a `Const` when both sides are numeric and the denominator is nonzero.


In [2]:
EXAMPLE_EXPRESSIONS = [
    "sin x",
    "pow x INT+ 2",
    "mul x mul INT+ 1 INT+ 2",
    "add sin x pow x INT+ 2",
]


def expr_tree_lines(node, indent: str = "") -> list[str]:
    if isinstance(node, Const):
        if node.is_named:
            return [f"{indent}Const(symbol={node.symbol!r})"]
        return [f"{indent}Const(value={node.value})"]

    if isinstance(node, Var):
        return [f"{indent}Var(name={node.name!r})"]

    if isinstance(node, UnaryOp):
        lines = [f"{indent}UnaryOp(op={node.op!r})", f"{indent}  operand:"]
        lines.extend(expr_tree_lines(node.operand, indent + "    "))
        return lines

    if isinstance(node, BinaryOp):
        lines = [f"{indent}BinaryOp(op={node.op!r})", f"{indent}  left:"]
        lines.extend(expr_tree_lines(node.left, indent + "    "))
        lines.append(f"{indent}  right:")
        lines.extend(expr_tree_lines(node.right, indent + "    "))
        return lines

    if isinstance(node, NaryOp):
        lines = [f"{indent}NaryOp(op={node.op!r}, operand_count={len(node.operands)})"]
        for index, operand in enumerate(node.operands):
            lines.append(f"{indent}  operand[{index}]:")
            lines.extend(expr_tree_lines(operand, indent + "    "))
        return lines

    raise TypeError(f"Unsupported expression type: {type(node).__name__}")


for expression in EXAMPLE_EXPRESSIONS:
    expr = parse_prefix_string(expression)
    roundtrip = serialize_prefix_string(expr)
    print("=" * 80)
    print(f"Expression:              {expression}")
    print(f"Parsed dataclass repr:   {expr!r}")
    print(f"Round-trip serialization:{roundtrip}")
    print("AST tree:")
    print("\n".join(expr_tree_lines(expr)))


Expression:              sin x
Parsed dataclass repr:   UnaryOp(token_start=0, token_end=2, op='sin', operand=Var(token_start=1, token_end=2, name='x'))
Round-trip serialization:sin x
AST tree:
UnaryOp(op='sin')
  operand:
    Var(name='x')
Expression:              pow x INT+ 2
Parsed dataclass repr:   BinaryOp(token_start=0, token_end=4, op='pow', left=Var(token_start=1, token_end=2, name='x'), right=Const(token_start=2, token_end=4, value=Fraction(2, 1), symbol=None))
Round-trip serialization:pow x INT+ 2
AST tree:
BinaryOp(op='pow')
  left:
    Var(name='x')
  right:
    Const(value=2)
Expression:              mul x mul INT+ 1 INT+ 2
Parsed dataclass repr:   NaryOp(token_start=0, token_end=7, op='mul', operands=(Var(token_start=1, token_end=2, name='x'), Const(token_start=3, token_end=5, value=Fraction(1, 1), symbol=None), Const(token_start=5, token_end=7, value=Fraction(2, 1), symbol=None)))
Round-trip serialization:mul x mul INT+ 1 INT+ 2
AST tree:
NaryOp(op='mul', operand_count=3

## Canonicalization And Position Indexing

`canonicalize(...)` is the first normalization pass applied before mutation. It does not try to prove arbitrary algebraic equivalence; it rewrites the AST into one consistent structural form that the mutation code can index and compare reliably.

Concretely, `canonicalize(...)`:

- normalizes operator and named-constant tokens,
- recursively canonicalizes children first,
- flattens nested associative `add` / `mul` nodes into one `NaryOp`,
- sorts operands of commutative n-ary operators by a structural key,
- folds numeric `div(const, const)` into a single `Const` when the denominator is nonzero,
- strips top-level additive constants that do not contain `x`, so `add INT+ 7 add x INT+ 2` becomes just `x`.

Mutation always starts from the canonical tree, not the raw parsed tree. That matters because node ids, token spans, subtree sizes, and equality checks are all defined on this normalized form.

`index_tree_positions(...)` then walks the canonical tree in preorder and records one `NodePosition` per node. Each row tells us:

- `node_id`: the preorder id later used by subtree replacement,
- `token_start` / `token_end`: the half-open span in the serialized canonical token sequence,
- `subtree_size`: the non-leaf size measure used by `sigma_small`,
- `production_family`: the family used to group compatible subtree replacements,
- `is_mutable`: whether the node has a local replacement and, if `sigma_small` is set, also fits under the subtree-size budget.

The spans shown below belong to the current canonical tree only. After a mutation, the tree must be indexed again to get fresh ids and spans for the new expression.


In [3]:
def print_table(rows: list[dict], columns: list[str]) -> None:
    if not rows:
        print("<no rows>")
        return

    widths = {
        column: max(len(column), max(len(str(row[column])) for row in rows))
        for column in columns
    }
    header = " | ".join(f"{column:<{widths[column]}}" for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(f"{str(row[column]):<{widths[column]}}" for column in columns))


def show_canonicalization(expression: str) -> None:
    original = parse_prefix_string(expression)
    canonical = canonicalize(original)
    print("-" * 80)
    print(f"Original:  {expression}")
    print(f"Canonical: {serialize_prefix_string(canonical)}")


for expression in EXAMPLE_EXPRESSIONS:
    show_canonicalization(expression)

print("-" * 80)
extra = "add INT+ 7 add x INT+ 2"
print(f"Extra simplification example: {extra}")
print(f"Canonical: {serialize_prefix_string(canonicalize(parse_prefix_string(extra)))}")

print("\nPosition index for 'add sin x pow x INT+ 2' with sigma_small=2:")
index_expr = canonicalize(parse_prefix_string("add sin x pow x INT+ 2"))
index = index_tree_positions(index_expr, sigma_small=2)
print(f"Serialized tokens: {tuple(serialize_prefix_string(index_expr).split())}")

rows = []
for position in index.positions:
    row = asdict(position)
    row["subtree"] = serialize_prefix_string(index.node_id_to_node[position.node_id])
    rows.append(row)

print_table(
    rows,
    [
        "node_id",
        "parent_id",
        "depth",
        "production_family",
        "op",
        "token_start",
        "token_end",
        "subtree_size",
        "is_mutable",
        "subtree",
    ],
)


--------------------------------------------------------------------------------
Original:  sin x
Canonical: sin x
--------------------------------------------------------------------------------
Original:  pow x INT+ 2
Canonical: pow x INT+ 2
--------------------------------------------------------------------------------
Original:  mul x mul INT+ 1 INT+ 2
Canonical: mul INT+ 1 mul INT+ 2 x
--------------------------------------------------------------------------------
Original:  add sin x pow x INT+ 2
Canonical: add sin x pow x INT+ 2
--------------------------------------------------------------------------------
Extra simplification example: add INT+ 7 add x INT+ 2
Canonical: x

Position index for 'add sin x pow x INT+ 2' with sigma_small=2:
Serialized tokens: ('add', 'sin', 'x', 'pow', 'x', 'INT+', '2')
node_id | parent_id | depth | production_family | op    | token_start | token_end | subtree_size | is_mutable | subtree               
--------+-----------+-------+---------------

## Local Same-Shape Replacement

Local replacement is the conservative mutation path. It changes the selected node in place without resampling its whole interior.

`local_replacement_candidates(node)` returns abstract replacement specs, not concrete AST nodes. For leaves, the spec says which leaf kind is allowed (`numeric_const`, `named_const`, or `var`). For operators, the spec fixes the replacement shape, operator label, and child count. A later step materializes one of those specs into an actual replacement expression.

`can_locally_replace(source, candidate)` enforces the exact local invariants:

- `Leaf <-> Leaf`
- `UnaryOp <-> UnaryOp`
- `BinaryOp <-> BinaryOp`
- `NaryOp(k) <-> NaryOp(k)`

For operator nodes, the children or operands must stay exactly the same and only the root label changes. For leaf nodes, the replacement must still be a leaf, must differ from the original node, and `Var` is limited to `x`. `BinaryOp` and `NaryOp(2)` do **not** cross locally, even if both happen to have two children, because local replacement preserves node class as well as arity.

`local_replace_once(expr, selected_node_id, rng)` canonicalizes the input, reindexes it, looks up the selected canonical node, samples one legal concrete replacement from the candidate specs, applies it by `node_id`, and canonicalizes the final tree again. Constant leaves are special here: a numeric or named-constant spec may be materialized into a nearby constant sampled from the constant banks, while a variable leaf can locally change only by turning into a constant leaf.


In [4]:
def format_spec(spec) -> str:
    if spec.leaf_kind is not None:
        return f"leaf:{spec.leaf_kind}"
    return f"{spec.shape}:{spec.op} (child_count={spec.child_count})"


def show_local_candidates(label: str, node) -> None:
    print("-" * 80)
    print(label)
    print(f"Node: {serialize_prefix_string(node)}")
    print([format_spec(spec) for spec in local_replacement_candidates(node)])


const_leaf = parse_prefix_string("INT+ 2")
var_leaf = parse_prefix_string("x")
unary_expr = parse_prefix_string("sin x")
binary_expr = parse_prefix_string("pow x INT+ 2")
nary_expr = NaryOp(
    op="mul",
    operands=(
        Var(name="x"),
        Const(value=Fraction(1, 1)),
        Const(value=Fraction(2, 1)),
    ),
)

show_local_candidates("Leaf constant candidates", const_leaf)
show_local_candidates("Variable x candidates", var_leaf)
show_local_candidates("Unary candidates", unary_expr)
show_local_candidates("Binary candidates", binary_expr)
show_local_candidates("N-ary candidates", nary_expr)

print("\nSelected can_locally_replace(...) checks:")
checks = [
    (
        "legal unary: sin(x) -> cos(x)",
        parse_prefix_string("sin x"),
        parse_prefix_string("cos x"),
    ),
    (
        "legal binary: pow(x, 2) -> div(x, 2)",
        parse_prefix_string("pow x INT+ 2"),
        parse_prefix_string("div x INT+ 2"),
    ),
    (
        "legal n-ary: mul(x, 1, 2) -> add(x, 1, 2)",
        nary_expr,
        NaryOp(op="add", operands=nary_expr.operands),
    ),
    (
        "illegal cross-shape: add(x, 1) -> pow(x, 1)",
        parse_prefix_string("add x INT+ 1"),
        parse_prefix_string("pow x INT+ 1"),
    ),
]

for label, source, target in checks:
    print("-" * 80)
    print(label)
    print(f"source: {serialize_prefix_string(source)}")
    print(f"target: {serialize_prefix_string(target)}")
    print(f"can_locally_replace: {can_locally_replace(source, target)}")


--------------------------------------------------------------------------------
Leaf constant candidates
Node: INT+ 2
['leaf:numeric_const', 'leaf:named_const', 'leaf:var']
--------------------------------------------------------------------------------
Variable x candidates
Node: x
['leaf:numeric_const', 'leaf:named_const']
--------------------------------------------------------------------------------
Unary candidates
Node: sin x
['unary:ln (child_count=1)', 'unary:exp (child_count=1)', 'unary:sqrt (child_count=1)', 'unary:abs (child_count=1)', 'unary:cos (child_count=1)', 'unary:tan (child_count=1)', 'unary:cot (child_count=1)', 'unary:sinh (child_count=1)', 'unary:cosh (child_count=1)', 'unary:tanh (child_count=1)', 'unary:coth (child_count=1)', 'unary:asin (child_count=1)', 'unary:acos (child_count=1)', 'unary:atan (child_count=1)', 'unary:acot (child_count=1)', 'unary:asinh (child_count=1)', 'unary:acosh (child_count=1)', 'unary:atanh (child_count=1)']
-------------------------

In [5]:
def show_local_mutation(label: str, expr, selected_node_id: int, seed: int) -> None:
    result = local_replace_once(expr, selected_node_id=selected_node_id, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"local_replace_once returned None for {label}")

    print("-" * 80)
    print(label)
    print(f"Canonical input expression: {serialize_prefix_string(canonicalize(expr))}")
    print(f"Selected node id:          {selected_node_id}")
    print(f"Original subtree:          {serialize_prefix_string(result.original_subtree)}")
    print(f"Replacement subtree:       {serialize_prefix_string(result.replacement_subtree)}")
    print(f"Final mutated expression:  {serialize_prefix_string(result.mutated_expr)}")


show_local_mutation(
    "Leaf replacement (constant leaf)",
    parse_prefix_string("pow x INT+ 5"),
    selected_node_id=2,
    seed=1,
)
show_local_mutation(
    "Unary replacement",
    parse_prefix_string("sin x"),
    selected_node_id=0,
    seed=1,
)
show_local_mutation(
    "Binary replacement",
    parse_prefix_string("pow x INT+ 2"),
    selected_node_id=0,
    seed=0,
)
show_local_mutation(
    "N-ary replacement",
    NaryOp(
        op="mul",
        operands=(
            Var(name="x"),
            Const(value=Fraction(1, 1)),
            Const(value=Fraction(2, 1)),
        ),
    ),
    selected_node_id=0,
    seed=0,
)


--------------------------------------------------------------------------------
Leaf replacement (constant leaf)
Canonical input expression: pow x INT+ 5
Selected node id:          2
Original subtree:          INT+ 5
Replacement subtree:       INT+ 6
Final mutated expression:  pow x INT+ 6
--------------------------------------------------------------------------------
Unary replacement
Canonical input expression: sin x
Selected node id:          0
Original subtree:          sin x
Replacement subtree:       cos x
Final mutated expression:  cos x
--------------------------------------------------------------------------------
Binary replacement
Canonical input expression: pow x INT+ 2
Selected node id:          0
Original subtree:          pow x INT+ 2
Replacement subtree:       div x INT+ 2
Final mutated expression:  div x INT+ 2
--------------------------------------------------------------------------------
N-ary replacement
Canonical input expression: mul INT+ 1 mul INT+ 2 x
Select

## Sampled Small Subtree Replacement

Sampled subtree replacement is the broader structural path. Instead of only changing the root label of the selected node, it can resample an entirely new subtree for the same production family and then splice that subtree into the canonical expression.

`sample_valid_subtree(family, sigma_small, rng)` is the constructor for these proposals. The `family` argument chooses the outer kind of subtree (`CONST`, `UNARY_EXPR`, `ADD_EXPR`, `MUL_EXPR`, `POW_EXPR`, `DIV_EXPR`, or the more general `EXPR` helper used internally), and `sigma_small` acts like a size budget for how much non-leaf structure may be generated beneath that root. Some families have extra rules: sampled `pow` subtrees often keep a constant exponent, and sampled `div` subtrees coerce away zero denominators.

`can_sampled_subtree_replace(source, candidate)` is looser than `can_locally_replace(...)`, but it still enforces production-family compatibility. A `pow` node can be replaced by another `pow` subtree with richer descendants, an `add` node by another `add`, and a numeric constant only by another numeric constant. Variables currently do not take the sampled subtree path.

`replace_subtree_by_node_id(expr, node_id, replacement)` does the actual tree surgery. It walks the tree using the same preorder numbering scheme as `index_tree_positions(...)` and swaps in the replacement when it reaches the requested node id. By itself, this function does not check whether the replacement is legal and it does not canonicalize the result; the caller is responsible for both of those steps.


In [8]:
original = parse_prefix_string("pow x sin x")
candidate = parse_prefix_string("pow x add x INT+ 1")
sampled_pow = sample_valid_subtree("POW_EXPR", sigma_small=2, rng=random.Random(0))

print(f"One sampled POW subtree proposal: {serialize_prefix_string(sampled_pow)}")
print(f"Original subtree:                 {serialize_prefix_string(original)}")
print(f"Candidate subtree:                {serialize_prefix_string(candidate)}")
print(f"can_sampled_subtree_replace:      {can_sampled_subtree_replace(original, candidate)}")

canonical_original = canonicalize(original)
mutated_raw = replace_subtree_by_node_id(canonical_original, 2, candidate)
mutated = canonicalize(mutated_raw)

print(f"Canonical original:               {serialize_prefix_string(canonical_original)}")
print(f"After replacement + canonicalize: {serialize_prefix_string(mutated)}")


One sampled POW subtree proposal: pow div INT+ 1 INT+ 2 x
Original subtree:                 pow x sin x
Candidate subtree:                pow x add x INT+ 1
can_sampled_subtree_replace:      True
Canonical original:               pow x sin x
After replacement + canonicalize: pow x pow x add INT+ 1 x


## Full Engine Walkthrough

`mutate_once(...)` is the full mutation engine. In the current implementation it performs this pipeline:

1. canonicalize the input expression,
2. index canonical positions with `index_tree_positions(...)`,
3. group mutable nodes by production family with `collect_candidate_nodes(...)`,
4. choose one family and one node from that family,
5. choose one mutation path for that node: `local_const_edit`, `local_same_arity_replacement`, or `sampled_small_subtree_replacement`,
6. build a concrete replacement subtree,
7. apply it with `replace_subtree_by_node_id(...)`,
8. canonicalize the mutated tree and reject no-op results.

`collect_candidate_nodes(...)` is the bridge between indexing and mutation selection: it runs canonicalization plus indexing and returns the mutable `NodePosition` records grouped by production family, which is exactly the pool `mutate_once(...)` samples from internally.

Two private helpers are worth knowing about when reading the source. `_sample_mutation_kind(...)` decides which of the three mutation paths are legal for the selected node, and `_apply_replacement(...)` performs the replacement plus the final canonicalization and no-op check.

The returned `MutationResult` records both what changed and where it changed in the pre-mutation canonical tree:

- `selected_node_id`: preorder id of the mutated node,
- `selected_family`: production family that was sampled,
- `selected_token_start` / `selected_token_end`: half-open token span in the canonical input serialization,
- `original_subtree`: the canonical subtree that was selected,
- `replacement_subtree`: the raw subtree proposal inserted before final canonicalization,
- `mutated_expr`: the final canonicalized expression returned to the caller.

Because of the final canonicalization step, `mutated_expr` can look more simplified or reordered than the raw `replacement_subtree` might suggest.


In [7]:
def summarize_candidate_pool(expression: str, sigma_small: int) -> dict[str, list[dict]]:
    canonical_expr = canonicalize(parse_prefix_string(expression))
    index = index_tree_positions(canonical_expr, sigma_small=sigma_small)
    candidates = collect_candidate_nodes(canonical_expr, sigma_small=sigma_small)
    return {
        family: [
            {
                "node_id": position.node_id,
                "span": (position.token_start, position.token_end),
                "subtree_size": position.subtree_size,
                "subtree": serialize_prefix_string(index.node_id_to_node[position.node_id]),
            }
            for position in positions
        ]
        for family, positions in sorted(candidates.items())
    }


def summarize_mutation_result(result) -> dict:
    return {
        "selected_node_id": result.selected_node_id,
        "selected_family": result.selected_family,
        "selected_token_start": result.selected_token_start,
        "selected_token_end": result.selected_token_end,
        "original_subtree": serialize_prefix_string(result.original_subtree),
        "replacement_subtree": serialize_prefix_string(result.replacement_subtree),
        "mutated_expr": serialize_prefix_string(result.mutated_expr),
    }


def run_mutate_once(expression: str, sigma_small: int, seed: int) -> None:
    candidate_pool = summarize_candidate_pool(expression, sigma_small)
    result = mutate_once(parse_prefix_string(expression), sigma_small=sigma_small, rng=random.Random(seed))
    if result is None:
        raise RuntimeError(f"mutate_once returned None for {expression}")

    print("-" * 80)
    print(f"Input expression: {expression}")
    print(f"sigma_small:      {sigma_small}")
    print(f"seed:             {seed}")
    print("Candidate pool by family:")
    pprint(candidate_pool)
    print("MutationResult:")
    pprint(summarize_mutation_result(result))


run_mutate_once("pow x INT+ 2", sigma_small=2, seed=0)
run_mutate_once("add sin x pow x INT+ 2", sigma_small=2, seed=0)


--------------------------------------------------------------------------------
Input expression: pow x INT+ 2
sigma_small:      2
seed:             0
Candidate pool by family:
{'CONST': [{'node_id': 2,
            'span': (2, 4),
            'subtree': 'INT+ 2',
            'subtree_size': 0}],
 'POW_EXPR': [{'node_id': 0,
               'span': (0, 4),
               'subtree': 'pow x INT+ 2',
               'subtree_size': 1}],
 'VAR': [{'node_id': 1, 'span': (1, 2), 'subtree': 'x', 'subtree_size': 0}]}
MutationResult:
{'mutated_expr': 'div x INT+ 2',
 'original_subtree': 'pow x INT+ 2',
 'replacement_subtree': 'div x INT+ 2',
 'selected_family': 'POW_EXPR',
 'selected_node_id': 0,
 'selected_token_end': 4,
 'selected_token_start': 0}
--------------------------------------------------------------------------------
Input expression: add sin x pow x INT+ 2
sigma_small:      2
seed:             0
Candidate pool by family:
{'CONST': [{'node_id': 5,
            'span': (5, 7),
         

## Closing Notes

- All mutation entry points operate on canonicalized trees, so equality, node ids, token spans, and subtree sizes are defined on normalized expressions rather than raw parser output.
- `index_tree_positions(...)` assigns preorder ids and canonical token spans, and `sigma_small` limits mutable nodes by `subtree_size`.
- Local replacement preserves node class, arity, and existing children or operands; only the root label or leaf value or kind changes.
- Sampled subtree replacement preserves production-family compatibility but can change internal descendant structure inside the sampled size budget.
- `MutationResult` points back to the selected location in the pre-mutation canonical tree, while the returned `mutated_expr` is canonicalized again and may therefore look reordered or simplified.
